# Download NASA turbofan data
Run the cell below in an Azure ML notebook. No AWS credentials or package installs are needed.

Source: A. Saxena and K. Goebel (2008), *Turbofan Engine Degradation Simulation Data Set*, NASA Ames Research Center.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

data_dir = Path("data/cmapss")
data_dir.mkdir(parents=True, exist_ok=True)
archive_path = data_dir / "turbofan.zip"

urlretrieve(
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
    archive_path,
)
with ZipFile(archive_path) as archive:
    archive.extractall(data_dir)

nested_zip = next(data_dir.rglob("CMAPSSData.zip"))
files_dir = data_dir / "CMAPSSData"
with ZipFile(nested_zip) as archive:
    archive.extractall(files_dir)

print(f"Files downloaded to: {files_dir.resolve()}")
print("\n".join(sorted(path.name for path in files_dir.iterdir())))

Files downloaded to: C:\Users\blaineperry\git\turbofan-pred-mx\notebooks\data\cmapss\CMAPSSData
Damage Propagation Modeling.pdf
RUL_FD001.txt
RUL_FD002.txt
RUL_FD003.txt
RUL_FD004.txt
readme.txt
test_FD001.txt
test_FD002.txt
test_FD003.txt
test_FD004.txt
train_FD001.txt
train_FD002.txt
train_FD003.txt
train_FD004.txt


## Register as an Azure ML data asset
After downloading, run this in a new cell. Replace the three workspace values. It uploads the extracted folder to your workspace storage as a `uri_folder` data asset (storage charges may apply).


In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import AzureAuthorityHosts, DefaultAzureCredential

client = MLClient.from_config(
    credential=DefaultAzureCredential(
        authority=AzureAuthorityHosts.AZURE_GOVERNMENT
    ),
    cloud="AzureUSGovernment",
)
asset = client.data.create_or_update(Data(
    name="nasa-cmapss",
    version="1",
    type="uri_folder",
    path=str(files_dir),
))
print(asset.id)


The asset appears under **Data** in Azure ML Studio. Use a new version number when registering a changed dataset. Use an Azure ML Python kernel with the Azure ML v2 SDK installed.